In [1]:
from pyspark.sql import SparkSession

In [2]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

from datetime import datetime
from datetime import timedelta
from dateutil.relativedelta import relativedelta

import logging

from IPython.core.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

/tmp/ipykernel_45/3320559344.py:10: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [3]:
spark = SparkSession.builder \
    .appName("mob_final") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .config("spark.cores.max", "15")\
    .config('spark.executor.memory', '7g')\
    .enableHiveSupport() \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/16 18:43:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
def build_final_mobile_features(spark, logical_date, logger):
    """Скрипт сборки абонентской базы для spfix_feature_store"""

    import pyspark.sql.functions as F

    business_month = datetime(logical_date.year, logical_date.month, 1).date()
    business_month_end = str(business_month + relativedelta(months=1, days= -1))

    logger.info('Building final dataframe...')
    logger.info(f'Logical date: {logical_date}')
    logger.info(f'Reporting date: {str(business_month)}')

    logger.info('Collecting all sources...')
    final_df = (
        spark.table('spfix_dm.mob_features_dm__1__base')
        .limit(700_000)
        .fillna(-1)
        .join(
            spark.table('spfix_dm.mob_features_dm__2__base'),
            ['msisdn'], 'left'
        )
        .fillna('unk')
        .join(
            spark.table('spfix_dm.mob_features_dm__3__base'),
            ['msisdn'], 'left'
        )
        .fillna(0)
        .join(
            spark.table('spfix_dm.mob_features_dm__4__base'),
            ['msisdn'], 'left'
        )
        .fillna(0)
        .join(
            spark.table('spfix_dm.mob_features_dm__5__base')
            .drop('app_n', 'regid'),
            ['msisdn'], 'left'
        )
        .fillna(0)
        .withColumn('build_dt', F.current_date())
        .withColumn('table_business_month', F.lit(business_month))
    )

    return final_df, business_month

In [5]:
log = logging.getLogger(__name__)
logical_date = datetime.now().date()

In [6]:
df, dt = build_final_mobile_features(spark, logical_date, log)

In [7]:
df

DataFrame[msisdn: int, app_n: int, regid: int, tp_change_dt: date, max_subs_fee_dt_day_31d: date, ep_tp_change_dt: date, device_first_imei_msisdn_dt: date, last_change_smartphone_dt: date, start_use_cur_smartp_mod_dt: date, mymts_max_activity_master_dt: date, mymts_max_activity_slave_dt: date, mnp_portation_in_mts_dt: date, last_active_roam_dt_day_90d: date, bill_group_name: int, lifetime: int, m_categ_name: int, m_reg_cd: int, m_reg_name: int, sale_channel_cd: int, sale_channel_name: int, fl_client_name_confirmed: int, client_type_cd: int, sex: int, age: int, citizenship_type: int, foreign_citizenship_cd: int, region_name: int, msisdn_type_cd: int, msisdn_cluster: int, trust_category: int, credit_limit_value: int, cnt_app_on_acc: int, service_provider_cd: int, fl_push_off: int, fl_unlim_internet: int, max_subs_fee_day_31d: decimal(10,2), host_type: int, activity_on_bts_dur_day_30d: int, cnt_give_out_prizes_day_7d: int, cnt_give_out_prizes_day_30d: int, cnt_give_out_prizes_day_90d: int

In [16]:
df.count()

2875000

In [8]:
df.count()

700000

In [9]:
(
    df
    .repartition(7)
    .write
    .mode("overwrite")
    .format("orc")
    .saveAsTable("spfix_dm.mob_features")
)

26/03/16 18:45:05 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/16 18:46:45 WARN TransportChannelHandler: Exception in connection from /172.18.0.8:53412
java.io.IOException: Connection reset by peer
	at java.base/sun.nio.ch.FileDispatcherImpl.read0(Native Method)
	at java.base/sun.nio.ch.SocketDispatcher.read(Unknown Source)
	at java.base/sun.nio.ch.IOUtil.readIntoNativeBuffer(Unknown Source)
	at java.base/sun.nio.ch.IOUtil.read(Unknown Source)
	at java.base/sun.nio.ch.IOUtil.read(Unknown Source)
	at java.base/sun.nio.ch.SocketChannelImpl.read(Unknown Source)
	at io.netty.buffer.PooledByteBuf.setBytes(PooledByteBuf.java:259)
	at io.netty.buffer.AbstractByteBuf.writeBytes(AbstractByteBuf.java:1132)
	at io.netty.channel.socket.nio.NioSocketChannel.doReadBytes(NioSocketChannel.java:357)
	at io.netty.channel.nio.AbstractNioByteChannel$NioByteUnsafe.read(AbstractNioByteC

Py4JJavaError: An error occurred while calling o78.saveAsTable.
: org.apache.spark.SparkException: Job aborted due to stage failure: Master removed our application: KILLED
	at org.apache.spark.scheduler.DAGScheduler.failJobAndIndependentStages(DAGScheduler.scala:2785)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2(DAGScheduler.scala:2721)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2$adapted(DAGScheduler.scala:2720)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreach$(ResizableArray.scala:55)
	at scala.collection.mutable.ArrayBuffer.foreach(ArrayBuffer.scala:49)
	at org.apache.spark.scheduler.DAGScheduler.abortStage(DAGScheduler.scala:2720)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1(DAGScheduler.scala:1206)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1$adapted(DAGScheduler.scala:1206)
	at scala.Option.foreach(Op

In [8]:
spark.table('spfix_dm.mob_features_dm__1__base').count()

1000000

In [7]:
(
    spark.table('spfix_dm.mob_features_dm__1__base')
    .fillna(-1)
    .join(
        spark.table('spfix_dm.mob_features_dm__2__base'),
        ['msisdn'], 'left'
    )
    .fillna('unk')
    .join(
        spark.table('spfix_dm.mob_features_dm__3__base'),
        ['msisdn'], 'left'
    )
    .fillna(0)
    .repartition(700)
    .join(
        spark.table('spfix_dm.mob_features_dm__4__base'),
        ['msisdn'], 'left'
    )
    .join(
        spark.table('spfix_dm.mob_features_dm__5__base')
        .drop('app_n', 'regid'),
        ['msisdn'], 'left'
    )
).count()

1000000

In [13]:
spark.catalog.refreshTable("spfix_dm.mob_features_dm__4__base")

In [21]:
spark.table("spfix_dm.mob_features_dm__4__base")

DataFrame[msisdn: int, home_diff_0_1: double, work_diff_0_1: double, home_diff_0_2: double, work_diff_0_2: double, home_diff_0_3: double, work_diff_0_3: double, home_diff_1_2: double, work_diff_1_2: double, home_diff_1_3: double, work_diff_1_3: double, home_diff_2_3: double, work_diff_2_3: double]

In [17]:
spark.stop()